In [ ]:

import random

import numpy as np

from brainrender import Scene
from brainrender.actors import Points

def get_n_random_points_in_region(region, N):
    """
    Gets N random points inside (or on the surface) of a mesh
    """

    region_bounds = region.mesh.bounds()
    X = np.random.randint(region_bounds[0], region_bounds[1], size=10000)
    Y = np.random.randint(region_bounds[2], region_bounds[3], size=10000)
    Z = np.random.randint(region_bounds[4], region_bounds[5], size=10000)
    pts = [[x, y, z] for x, y, z in zip(X, Y, Z)]

    ipts = region.mesh.inside_points(pts).coordinates
    return np.vstack(random.choices(ipts, k=N))




# Display the Allen Brain mouse atlas.
scene = Scene(atlas_name="allen_mouse_25um", title="Cells in primary visual cortex")

# Display a brain region
primary_visual = scene.add_brain_region("VISp", alpha=0.2)

# Get a numpy array with (fake) coordinates of some labelled cells
coordinates = get_n_random_points_in_region(primary_visual, 2000)

# Create a Points actor
cells = Points(coordinates)

# Add to scene
scene.add(cells)

# Add label to the brain region
scene.add_label(primary_visual, "Primary visual cortex")

# Display the figure.
scene.render()

scene.export("brain_regions.html")



In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import re
from brainrender import Scene  # Ensure Scene is imported if not already

# Load the XML
tree = ET.parse("3-12.xml")
root = tree.getroot()
slice_element = root.find("slice")

# Extract anchoring string
anchoring = slice_element.attrib['anchoring']

# Parse all the variables using regex
numbers = dict(re.findall(r'(\w+)=([-+eE0-9.]+)', anchoring))
# Convert string values to floats
for key in numbers:
    numbers[key] = float(numbers[key])

# Extract origin (slice center in CCF)
ccf_x = numbers['ox']
ccf_y = numbers['oy']
ccf_z = numbers['oz']

# Extract u and v vectors
u = np.array([numbers['ux'], numbers['uy'], numbers['uz']])
v = np.array([numbers['vx'], numbers['vy'], numbers['vz']])

# Compute normal vector
normal = np.cross(u, v)
normal = normal / np.linalg.norm(normal)  # normalize

# Output
print(f"Slice center in CCF (µm): x={ccf_x:.2f}, y={ccf_y:.2f}, z={ccf_z:.2f}")
print(f"Normal vector: {normal}")

# Display the Allen Brain mouse atlas.
# Added jupyter_backend for inline notebook rendering and updated title
scene = Scene(atlas_name="allen_mouse_25um", title="Cells in primary visual cortex")
# Display a brain region
primary_visual = scene.add_brain_region("VISp", alpha=0.2)

# Define the position for the plane from your XML data
slice_position = (ccf_x, ccf_y, ccf_z)

# Create the custom plane using the scene's atlas
custom_plane = scene.atlas.get_plane(pos=slice_position, norm=normal)

# Debug: print types to ensure correctness
print(f"custom_plane type: {type(custom_plane)}")
print(f"primary_visual type: {type(primary_visual)}")

# Use scene.slice() to draw the plane, slicing the primary_visual actor
scene.slice(custom_plane, actors=[primary_visual])

# Add label to the brain region
scene.add_label(primary_visual, "Primary visual cortex (sliced)")

# Display the figure.
scene.render()

scene.export("slice.html")


In [34]:
from pathlib import Path

from myterial import orange
from rich import print

from brainrender import Scene

print(f"[{orange}]Running example: slicing a brain region[/]")


# Create a brainrender scene
scene = Scene(title="slice")

# Add brain regions
th = scene.add_brain_region("TH")

# You can specify color, transparency...
mos, ca1 = scene.add_brain_region("MOs", "CA1", alpha=0.2, color="green")

# Slice with a custom plane
plane = scene.atlas.get_plane(pos=mos.center_of_mass(), norm=(1, 1, 0))
scene.slice(plane, actors=[mos, ca1])

# Render!
scene.render()
scene.export("slice_brain_regions.html")

Running example: slicing a brain region

The brainrender scene has been exported for web. The results are saved at slice_brain_regions.html

'slice_brain_regions.html'